# ASL Sign Language Translator — Training Notebook

This notebook walks through the full pipeline:
1. Dataset exploration
2. Preprocessing
3. Model training (MobileNetV2)
4. Evaluation & visualization


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')  # Project root

import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from collections import Counter
import cv2

print(f'TF version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')


## 1. Dataset Info & Download Links

### Recommended Datasets

| Dataset | Size | Format | Link |
|---------|------|--------|------|
| **Kaggle ASL Alphabet** | 87,000 images, 200×200 px, 29 classes | RGB JPEG | [kaggle.com/grassknoted/asl-alphabet](https://www.kaggle.com/grassknoted/asl-alphabet) |
| **ASL MNIST** | 27,455 train / 7,172 test, 28×28 grayscale, 24 classes | CSV | [kaggle.com/datamunge/sign-language-mnist](https://www.kaggle.com/datamunge/sign-language-mnist) |
| **Roboflow ASL** | 6,285 annotated images | YOLO/VOC | [roboflow.com](https://universe.roboflow.com/david-lee-d0rhs/american-sign-language-letters) |

### Download (Kaggle API)
```bash
pip install kaggle
kaggle datasets download grassknoted/asl-alphabet -p data/raw/ --unzip
```

### Expected folder structure after download:
```
data/raw/
  asl_alphabet_train/
    A/  *.jpg
    B/  *.jpg
    ...
    Z/  *.jpg
```


In [ ]:
# ── Explore class distribution ─────────────────────────────────────
DATA_DIR = Path('../data/raw/asl_alphabet_train')  # adjust if needed

class_counts = {}
for cls_dir in sorted(DATA_DIR.iterdir()):
    if cls_dir.is_dir() and cls_dir.name.upper() in list('ABCDEFGHIJKLMNOPQRSTUVWXYZ'):
        n = len(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
        class_counts[cls_dir.name.upper()] = n

fig, ax = plt.subplots(figsize=(16, 4))
labels, counts = zip(*sorted(class_counts.items()))
bars = ax.bar(labels, counts, color='#00e5a0', edgecolor='#00a070', linewidth=0.8)
ax.set_title('ASL Dataset — Samples per Class', fontsize=14)
ax.set_xlabel('Class (Letter)')
ax.set_ylabel('Count')
ax.axhline(min(counts), color='red', linestyle='--', label=f'Min: {min(counts)}')
ax.axhline(max(counts), color='blue', linestyle='--', label=f'Max: {max(counts)}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Total images: {sum(counts):,}')
print(f'Classes: {len(counts)}')
print(f'Min/Max per class: {min(counts)} / {max(counts)}')


In [ ]:
# ── Preview sample images ──────────────────────────────────────────
fig, axes = plt.subplots(4, 7, figsize=(16, 10))
axes = axes.flatten()

for i, (label, files) in enumerate(sorted(class_counts.items())):
    if i >= 26: break
    sample = random.choice(list((DATA_DIR / label).glob('*.jpg')))
    img = cv2.imread(str(sample))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(label, fontsize=14, fontweight='bold')
    axes[i].axis('off')

# Hide any unused subplots
for j in range(i+1, len(axes)): axes[j].axis('off')

plt.suptitle('ASL Alphabet Sample Images', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Run preprocessing pipeline ────────────────────────────────────
# This calls the preprocessing script from the scripts/ directory
import subprocess
result = subprocess.run(
    ['python', '../scripts/preprocess.py',
     '--data_dir', '../data/raw/asl_alphabet_train',
     '--output_dir', '../data/processed'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


In [ ]:
# ── Run training ──────────────────────────────────────────────────
result = subprocess.run(
    ['python', '../scripts/train.py',
     '--data_dir',       '../data/processed',
     '--model_dir',      '../model/saved',
     '--checkpoint_dir', '../model/checkpoints',
     '--phase1_epochs',  '10',
     '--phase2_epochs',  '20'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])   # Last 3000 chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


In [ ]:
# ── View training history ─────────────────────────────────────────
from IPython.display import Image, display
display(Image('../model/saved/training_history.png'))
display(Image('../model/saved/confusion_matrix.png'))


In [ ]:
# ── Load model and test on a single image ─────────────────────────
import sys; sys.path.insert(0, '..')
from backend.utils.predictor import ASLPredictor

predictor = ASLPredictor(
    '../model/saved/asl_model_final.keras',
    '../model/saved/label_map.json'
)

# Test on a random image from the test set
test_dir = Path('../data/processed/test')
test_imgs = list(test_dir.glob('**/*.jpg'))[:5]

fig, axes = plt.subplots(1, len(test_imgs), figsize=(15, 4))
for ax, img_path in zip(axes, test_imgs):
    true_label = img_path.parent.name
    img = cv2.imread(str(img_path))
    result = predictor.predict(img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    color = 'green' if result['letter'] == true_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {result['letter']} ({result['confidence']:.0%})",
                 color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Model Predictions on Test Samples', fontsize=13)
plt.tight_layout()
plt.show()
